# 01 · Data Loading & Cleaning

**Purpose:** Pull raw data from PostgreSQL, clean it, and persist to parquet.

**Outputs:** `companies.parquet`, `stock_prices.parquet`, `articles.parquet`,
`sentiments.parquet`, `company_mentions.parquet`, `sector_mentions.parquet`

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

# Make utils importable from this notebook's directory
NB_DIR = Path().resolve()
sys.path.insert(0, str(NB_DIR))
from utils import query, save, DATA_DIR

print(f"PostgreSQL data  →  {DATA_DIR}")

PostgreSQL data  →  C:\_PROJECTS\pfa_bvc\Notebooks\signal_pipeline\data


## 1 · Companies

In [2]:
def load_companies() -> pd.DataFrame:
    return query("""
        SELECT ticker, company_name, sector, parent,
               description, ceo, founded, headquarters,
               revenue, employees, stock_exchange, siege_social
        FROM companies
        ORDER BY ticker
    """)

companies = load_companies()
print(f"{len(companies)} companies")
companies.head()

72 companies


C:\_PROJECTS\pfa_bvc\Notebooks\signal_pipeline\utils.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


,ticker,company_name,sector,parent,description,ceo,founded,headquarters,revenue,employees,stock_exchange,siege_social
0,ADH,Addoha Group,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ADI,Alliances Développement Immobilier,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFI,Afric Industries,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFM,AFMA,Insurance,Independent,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AGM,AGMA,Insurance,Independent,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2 · Stock Prices

In [3]:
def load_stock_prices() -> pd.DataFrame:
    df = query("""
        SELECT ticker, date, close, open, high, low, volume, change_pct
        FROM stock_prices
        ORDER BY ticker, date
    """)
    df["date"] = pd.to_datetime(df["date"])
    df[["close", "open", "high", "low", "volume", "change_pct"]] = (
        df[["close", "open", "high", "low", "volume", "change_pct"]].apply(pd.to_numeric, errors="coerce")
    )
    return df

stock_prices = load_stock_prices()
print(f"{len(stock_prices):,} rows  |  {stock_prices['ticker'].nunique()} tickers")
print(f"Date range: {stock_prices['date'].min().date()} → {stock_prices['date'].max().date()}")
stock_prices.head()

C:\_PROJECTS\pfa_bvc\Notebooks\signal_pipeline\utils.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


42,702 rows  |  46 tickers
Date range: 2021-01-04 → 2026-05-13


,ticker,date,close,open,high,low,volume,change_pct
0,ADH,2021-01-04,6.45,6.59,6.59,6.36,170680.0,0.94
1,ADH,2021-01-05,6.57,6.49,6.60,6.45,93820.0,1.86
2,ADH,2021-01-06,6.72,6.57,6.80,6.56,367630.0,2.28
3,ADH,2021-01-07,6.70,6.74,6.79,6.65,293190.0,-0.30
4,ADH,2021-01-08,6.65,6.74,6.77,6.56,25200.0,-0.75


## 3 · Articles

In [4]:
def clean_text(text) -> str:
    """Strip HTML tags and collapse whitespace."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def load_articles() -> pd.DataFrame:
    df = query("""
        SELECT a.url, a.title, a.published_at, a.full_text,
               a.language, a.feed_url, f.name AS feed_name
        FROM   articles a
        LEFT JOIN feeds f ON f.url = a.feed_url
        WHERE  a.published_at IS NOT NULL
    """)
    df["published_at"] = pd.to_datetime(df["published_at"], utc=True)
    # Timezone-naive calendar date for joining with prices
    df["date"] = df["published_at"].dt.tz_localize(None).dt.normalize()
    df["title"]     = df["title"].apply(clean_text)
    df["full_text"] = df["full_text"].apply(clean_text)
    # Combined text for matching and event detection
    df["text"] = (df["title"].fillna("") + " " + df["full_text"].fillna("")).str.strip()
    df = df.drop_duplicates(subset=["url"])
    return df

articles = load_articles()
print(f"{len(articles):,} articles")
print(f"Date range: {articles['date'].min().date()} → {articles['date'].max().date()}")
print(f"Languages: {articles['language'].value_counts().to_dict()}")
articles[["url", "title", "date", "language", "feed_name"]].head()

C:\_PROJECTS\pfa_bvc\Notebooks\signal_pipeline\utils.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


974 articles
Date range: 2006-09-20 → 2026-04-24
Languages: {'fr': 745, 'ar': 229}


,url,title,date,language,feed_name
0,https://lematin.ma/societe/les-cyberattaques-s...,Les cyberattaques se multiplient au Maroc : un...,2026-04-19,fr,LeMatin Economie
1,https://lematin.ma/-Marrakech-_L-Agence-urbain...,L’Agence urbaine met en place une nouvelle str...,2013-04-24,fr,Backfill: lematin
2,https://lematin.ma/.../Activites-Royales.../36...,Création de la nouvelle ville de Tamesna,2007-03-13,fr,Backfill: lematin
3,https://www.hespress.com/%d8%a5%d8%b4%d8%a7%d8...,إشادة كندية بالأمن الرياضي المغربي,2026-04-19,ar,Hespress Arabic
4,https://lematin.ma/.../alaa-halifi-parle.../37...,"Éloge de la folie, de Halifi, immersion dans l...",2022-04-22,fr,Backfill: lematin


## 4 · Sentiment Scores

In [5]:
def load_sentiments() -> pd.DataFrame:
    df = query("""
        SELECT article_url, sentiment, score, confidence, reasoning, analyzed_at
        FROM   sentiment_scores
    """)
    df["analyzed_at"] = pd.to_datetime(df["analyzed_at"], utc=True)
    df["score"]       = pd.to_numeric(df["score"], errors="coerce")
    df["confidence"]  = pd.to_numeric(df["confidence"], errors="coerce")
    return df

sentiments = load_sentiments()
print(f"{len(sentiments):,} sentiment scores")
sentiments["sentiment"].value_counts()

C:\_PROJECTS\pfa_bvc\Notebooks\signal_pipeline\utils.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


709 sentiment scores


sentiment
neutral     545
positive    120
negative     44
Name: count, dtype: int64

## 5 · Entity Mentions

In [6]:
def load_company_mentions() -> pd.DataFrame:
    return query("""
        SELECT m.article_url, m.ticker, c.company_name, c.sector
        FROM   article_company_mentions m
        LEFT JOIN companies c ON c.ticker = m.ticker
    """)

def load_sector_mentions() -> pd.DataFrame:
    return query("""
        SELECT article_url, sector_name
        FROM   article_sector_mentions
    """)

company_mentions = load_company_mentions()
sector_mentions  = load_sector_mentions()
print(f"Company mentions: {len(company_mentions):,}")
print(f"Sector mentions:  {len(sector_mentions):,}")
company_mentions.head()

C:\_PROJECTS\pfa_bvc\Notebooks\signal_pipeline\utils.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn, params=params)


Company mentions: 61
Sector mentions:  90


,article_url,ticker,company_name,sector
0,https://lematin.ma/economie/la-bei-accelere-se...,ADI,Alliances Développement Immobilier,NaN
1,https://lematin.ma/economie/la-bei-accelere-se...,ATL,AtlantaSanad,Insurance
2,https://www.lavieeco.com/affaires/biscuiterie-...,AFI,Afric Industries,NaN
3,https://www.lavieeco.com/comese/srm-casablanca...,SRM,Societe de Réalisations Mécaniques,NaN
4,https://lematin.ma/10-ans-capacites-mondiales-...,BCI,BMCI (Banque Marocaine pour le Commerce et l’I...,Banking


## 6 · Missing-value summary

In [7]:
def missing_report(df: pd.DataFrame, label: str) -> None:
    missing = df.isna().mean().mul(100).round(1)
    missing = missing[missing > 0]
    if missing.empty:
        print(f"[{label}] no missing values")
    else:
        print(f"[{label}] columns with missing values (%):\n{missing.to_string()}\n")

missing_report(stock_prices, "stock_prices")
missing_report(articles[["url", "title", "full_text", "language"]], "articles")
missing_report(sentiments[["article_url", "sentiment", "score"]], "sentiments")

[stock_prices] no missing values
[articles] no missing values
[sentiments] no missing values


## 7 · Save

In [8]:
save(companies,        "companies")
save(stock_prices,     "stock_prices")
save(articles,         "articles")
save(sentiments,       "sentiments")
save(company_mentions, "company_mentions")
save(sector_mentions,  "sector_mentions")

  saved 72 rows  →  companies.parquet
  saved 42,702 rows  →  stock_prices.parquet
  saved 974 rows  →  articles.parquet
  saved 709 rows  →  sentiments.parquet
  saved 61 rows  →  company_mentions.parquet
  saved 90 rows  →  sector_mentions.parquet


WindowsPath('C:/_PROJECTS/pfa_bvc/Notebooks/signal_pipeline/data/sector_mentions.parquet')